In [2]:
import os
import json
import pandas as pd
import numpy as np
from typing import Iterator
from pathlib import Path

In [112]:
df_accidents = pd.read_csv('../../data/intermediate/incidents.csv')
df_accidents['datetime'] = pd.to_datetime(df_accidents.datetime)
df_accidents

,id,datetime,longitude,latitude,voivodeship,county,municipality,city,incident_type,other_cause,...,site_characteristic,speed_limit,road_type,road_geometry,pavement,surface_condition,surface_marking,intersection_type,traffic_lights,built_up_area
0,98772347,2015-02-23 15:55:00,19.970528,50.079028,małopolskie,Kraków,Kraków,Kraków,Najechanie na pieszego,Brak,...,Przejście dla pieszych,50.0,Dwie jezdnie jednokierunkowe,Brak,Twarda,Sucha,Jest,Z drogą z pierwsz.,"Jest, działa",Obszar zabudowany
1,110531047,2021-09-21 06:35:00,22.497722,52.911667,podlaskie,wysokomazowiecki,Wysokie Mazowieckie,Wysokie Mazowieckie,Najechanie na pieszego,Brak,...,Przejście dla pieszych,50.0,Jednojezdniowa dwukierunkowa,Odcinek prosty,Twarda,Sucha,Jest,Brak,Brak,Obszar zabudowany
2,104988338,2016-11-01 19:30:00,23.161944,53.137778,podlaskie,Białystok,Białystok,Białystok,Najechanie na pieszego,Brak,...,Przejście dla pieszych,50.0,Jednojezdniowa dwukierunkowa,Brak,Twarda,Mokra,Jest,Z drogą z pierwsz.,Brak,Obszar zabudowany
3,124590550,2023-01-13 09:00:00,18.617166,54.375527,pomorskie,Gdańsk,Gdańsk,Gdańsk,Zderzenie pojazdów tylne,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,104551628,2016-06-30 13:00:00,17.467750,50.856611,opolskie,brzeski,Brzeg,Brzeg,Zderzenie pojazdów boczne,Brak,...,Jezdnia,50.0,Jednojezdniowa dwukierunkowa,Brak,Twarda,Sucha,Jest,Z drogą z pierwsz.,Brak,Obszar zabudowany
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249952,120878012,2022-07-05 13:24:00,22.051306,49.959167,podkarpackie,rzeszowski,Tyczyn,Tyczyn,Wywrócenie się pojazdu,Brak,...,Jezdnia,50.0,Jednojezdniowa dwukierunkowa,Odcinek prosty,Twarda,Sucha,Jest,Brak,Brak,Obszar zabudowany
249953,109491179,2020-10-26 18:25:00,18.595000,54.404444,pomorskie,Gdańsk,Gdańsk,Gdańsk,Zderzenie pojazdów boczne,Brak,...,"Droga, pas ruchu, śluza dla rowerów",50.0,Jednojezdniowa dwukierunkowa,Odcinek prosty,Twarda,Sucha,Nie ma,Brak,Brak,Obszar zabudowany
249954,109259104,2020-07-03 17:32:00,17.067916,50.779471,dolnośląskie,strzeliński,Strzelin,Strzelin,Najechanie na pieszego,Brak,...,Przejście dla pieszych,50.0,Dwie jezdnie jednokierunkowe,Brak,Twarda,Sucha,Jest,Brak,Brak,Obszar zabudowany
249955,106049857,2017-06-15 13:10:00,18.587500,54.358611,pomorskie,Gdańsk,Gdańsk,Gdańsk,Wypadek z pasażerem,Nieustalone,...,Jezdnia,50.0,Dwie jezdnie jednokierunkowe,Brak,Twarda,Sucha,Jest,Z drogą z pierwsz.,Brak,Obszar zabudowany


## Add elevation

In [113]:
# https://ec.europa.eu/eurostat/web/gisco/geodata/digital-elevation-model/copernicus#Elevation

import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.warp import transform_bounds, transform
from rasterio.mask import mask
from shapely.geometry import box

# Load DEM data functions
def load_dem_data(dem_filepath):
    """
    Load Digital Elevation Model data and create an interpolation function
    """
    with rasterio.open(dem_filepath) as dem:
        elevation = dem.read(1)
        transform = dem.transform
        bounds = dem.bounds
        
        # rows, cols = elevation.shape
        # xs = np.linspace(bounds.left, bounds.right, cols)
        # ys = np.linspace(bounds.bottom, bounds.top, rows)
        
        return elevation, transform, bounds

def get_elevation_for_point(lat, lon, elevation, transform):
    """
    Get elevation for a specific lat/lon point
    """
    row, col = ~transform * (lon, lat)
    row, col = int(row), int(col)
    
    if (0 <= row < elevation.shape[0] and 
        0 <= col < elevation.shape[1]):
        return elevation[col, row]
    return np.nan

# Load the DEM data
dem_filepath = '../../data/input/geographic/poland_dem.tif'
elevation_data, dem_transform, dem_bounds = load_dem_data(dem_filepath)

In [114]:
df_accidents['elevation'] = df_accidents.apply(lambda row: get_elevation_for_point(row['latitude'], row['longitude'], elevation_data, dem_transform), axis=1)
df_accidents

,id,datetime,longitude,latitude,voivodeship,county,municipality,city,incident_type,other_cause,...,speed_limit,road_type,road_geometry,pavement,surface_condition,surface_marking,intersection_type,traffic_lights,built_up_area,elevation
0,98772347,2015-02-23 15:55:00,19.970528,50.079028,małopolskie,Kraków,Kraków,Kraków,Najechanie na pieszego,Brak,...,50.0,Dwie jezdnie jednokierunkowe,Brak,Twarda,Sucha,Jest,Z drogą z pierwsz.,"Jest, działa",Obszar zabudowany,208.385101
1,110531047,2021-09-21 06:35:00,22.497722,52.911667,podlaskie,wysokomazowiecki,Wysokie Mazowieckie,Wysokie Mazowieckie,Najechanie na pieszego,Brak,...,50.0,Jednojezdniowa dwukierunkowa,Odcinek prosty,Twarda,Sucha,Jest,Brak,Brak,Obszar zabudowany,145.495544
2,104988338,2016-11-01 19:30:00,23.161944,53.137778,podlaskie,Białystok,Białystok,Białystok,Najechanie na pieszego,Brak,...,50.0,Jednojezdniowa dwukierunkowa,Brak,Twarda,Mokra,Jest,Z drogą z pierwsz.,Brak,Obszar zabudowany,130.480957
3,124590550,2023-01-13 09:00:00,18.617166,54.375527,pomorskie,Gdańsk,Gdańsk,Gdańsk,Zderzenie pojazdów tylne,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.598199
4,104551628,2016-06-30 13:00:00,17.467750,50.856611,opolskie,brzeski,Brzeg,Brzeg,Zderzenie pojazdów boczne,Brak,...,50.0,Jednojezdniowa dwukierunkowa,Brak,Twarda,Sucha,Jest,Z drogą z pierwsz.,Brak,Obszar zabudowany,151.008774
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249952,120878012,2022-07-05 13:24:00,22.051306,49.959167,podkarpackie,rzeszowski,Tyczyn,Tyczyn,Wywrócenie się pojazdu,Brak,...,50.0,Jednojezdniowa dwukierunkowa,Odcinek prosty,Twarda,Sucha,Jest,Brak,Brak,Obszar zabudowany,230.263031
249953,109491179,2020-10-26 18:25:00,18.595000,54.404444,pomorskie,Gdańsk,Gdańsk,Gdańsk,Zderzenie pojazdów boczne,Brak,...,50.0,Jednojezdniowa dwukierunkowa,Odcinek prosty,Twarda,Sucha,Nie ma,Brak,Brak,Obszar zabudowany,10.589193
249954,109259104,2020-07-03 17:32:00,17.067916,50.779471,dolnośląskie,strzeliński,Strzelin,Strzelin,Najechanie na pieszego,Brak,...,50.0,Dwie jezdnie jednokierunkowe,Brak,Twarda,Sucha,Jest,Brak,Brak,Obszar zabudowany,165.091919
249955,106049857,2017-06-15 13:10:00,18.587500,54.358611,pomorskie,Gdańsk,Gdańsk,Gdańsk,Wypadek z pasażerem,Nieustalone,...,50.0,Dwie jezdnie jednokierunkowe,Brak,Twarda,Sucha,Jest,Z drogą z pierwsz.,Brak,Obszar zabudowany,94.792969


## Add weather data

### Prepare weather data for each station

In [115]:
df_weather_stations = pd.read_csv('../../data/intermediate/weather_stations.csv')

mask = df_weather_stations['elevation'].isna()
df_weather_stations.loc[mask, 'elevation'] = df_weather_stations.loc[mask].apply(
    lambda row: get_elevation_for_point(row['latitude'], row['longitude'], elevation_data, dem_transform), axis=1
)
df_weather_stations.head()

,id,name,latitude,longitude,elevation,temperature_start_date,precipitation_start_date,humidity_start_date,category
0,249170080,Dzierżkowice,49.992222,17.846389,270.0,NaN,NaN,NaN,No data
1,249180010,Pszczyna,49.995556,18.919167,270.0,2015-01,2019-04,2015-01,"Precipitation, Temperature, Humidity"
2,249180020,Warszowice,49.991944,18.705556,270.0,NaN,2015-01,NaN,Precipitation only
3,249180030,Chałupki,49.925833,18.327500,198.0,NaN,NaN,NaN,No data
4,249180070,Mazańcowice,49.862500,18.969444,269.0,NaN,2015-01,NaN,Precipitation only


In [116]:
df_parameters = pd.read_csv('../../data/input/weather/kody_parametr.csv', encoding='windows-1250', sep=';')
df_parameters

,Kod,Nazwa
0,B00300S,Temperatura powietrza (oficjalna)
1,B00305A,Temperatura gruntu (czujnik)
2,B00202A,Kierunek wiatru (czujnik)
3,B00702A,Średnia prędkość wiatru czujnik 10 minut
4,B00703A,Prędkość maksymalna (czujnik)
5,B00608S,Suma opadu 10 minutowego
6,B00604S,Suma opadu dobowego
7,B00606S,Suma opadu godzinowego
8,B00802A,Wilgotność względna powietrza (czujnik)
9,B00714A,Największy poryw w okresie 10min ze stacji Syn...


In [117]:
params = df_parameters.loc[[0, 7, 8], 'Kod'].tolist()
params

['B00300S', 'B00606S', 'B00802A']

In [118]:
def get_measurements_df(param: str, year: int, month: int) -> pd.DataFrame:
    measurement_path = Path('../../data/input/weather/') / str(year) / str(month).zfill(2) / f'{param}_{year}_{str(month).zfill(2)}.csv'
    columns = ['station_id', 'measurement', 'datetime', 'value']
    df_measurements = pd.read_csv(measurement_path, sep=';', encoding='windows-1250', names=columns, header=None, index_col=False, dtype={'station_id': str, 'value': str, 'datetime': str, 'measurement': str})

    df_measurements['value'] = df_measurements['value'].str.replace(',', '.').astype(float)
    df_measurements['datetime'] = pd.to_datetime(df_measurements['datetime'])

    df_measurements['station_id'] = pd.to_numeric(df_measurements['station_id'], errors='coerce').astype('Int64')
    df_measurements.dropna(subset=['station_id'], inplace=True)

    return df_measurements

In [122]:
from functools import reduce

param_map = {'B00300S': 'temperature', 'B00606S': 'precipation', 'B00802A': 'humidity'}

merged_dfs = []
for param, param_name in param_map.items():
    temp_list = []
    for year in range(2015, 2021):
        for month in range(1, 13):
            df_meas = get_measurements_df(param, year, month)
            temp_list.append(df_meas)
    param_df = pd.concat(temp_list, ignore_index=True)
    param_df.rename(columns={'value': param_name}, inplace=True)
    param_df.drop(columns=['measurement'], inplace=True)
    merged_dfs.append(param_df)

df_weather_measurements = reduce(lambda left, right: pd.merge(left, right, on=['datetime', 'station_id'], how='outer'), merged_dfs)
df_weather_measurements.sort_values(by=['station_id','datetime'], inplace=True)
df_weather_measurements

,station_id,datetime,temperature,precipation,humidity
0,249180010,2015-01-01 00:00:00,-3.5,NaN,74.0
469,249180010,2015-01-01 00:10:00,-3.5,NaN,74.0
702,249180010,2015-01-01 00:20:00,-3.4,NaN,75.0
933,249180010,2015-01-01 00:30:00,-3.4,NaN,75.0
1164,249180010,2015-01-01 00:40:00,-3.4,NaN,75.0
...,...,...,...,...,...
89133826,354220195,2020-12-31 23:10:00,-1.8,NaN,100.0
89134070,354220195,2020-12-31 23:20:00,-1.8,NaN,100.0
89134313,354220195,2020-12-31 23:30:00,-1.9,NaN,100.0
89134556,354220195,2020-12-31 23:40:00,-2.0,NaN,100.0


In [123]:
# Create a copy of datetime rounded to hours for matching
df_weather_measurements['hour'] = df_weather_measurements['datetime'].dt.floor('h')

# Create a temporary dataframe with only precipitation data at hourly intervals
precip_hourly = df_weather_measurements[['station_id', 'hour', 'precipation']].dropna()
precip_hourly = precip_hourly.drop_duplicates(['station_id', 'hour'])

# Merge the hourly precipitation back to the main dataframe
df_weather_measurements = df_weather_measurements.merge(
    precip_hourly,
    on=['station_id', 'hour'],
    how='left',
    suffixes=('_orig', '')
)

# Clean up
df_weather_measurements.drop(columns=['precipation_orig', 'hour'], inplace=True)
df_weather_measurements

,station_id,datetime,temperature,humidity,precipation
0,249180010,2015-01-01 00:00:00,-3.5,74.0,NaN
1,249180010,2015-01-01 00:10:00,-3.5,74.0,NaN
2,249180010,2015-01-01 00:20:00,-3.4,75.0,NaN
3,249180010,2015-01-01 00:30:00,-3.4,75.0,NaN
4,249180010,2015-01-01 00:40:00,-3.4,75.0,NaN
...,...,...,...,...,...
89134796,354220195,2020-12-31 23:10:00,-1.8,100.0,0.0
89134797,354220195,2020-12-31 23:20:00,-1.8,100.0,0.0
89134798,354220195,2020-12-31 23:30:00,-1.9,100.0,0.0
89134799,354220195,2020-12-31 23:40:00,-2.0,100.0,0.0


In [124]:
df_weather_measurements.sort_values(['station_id', 'datetime'], inplace=True)

# Set the datetime as the index (required for a time-based rolling window)
df_weather_measurements.set_index('datetime', inplace=True)

# Calculate the 24h rolling sum for each station.
# Note: The '24h' window will sum all available precipitation values in the previous 24 hours.
df_weather_measurements['precipation_24h'] = (
    df_weather_measurements.groupby('station_id')['precipation']
    .rolling('24h').sum()
    .reset_index(level=0, drop=True)
)

df_weather_measurements.reset_index(inplace=True)
df_weather_measurements

,datetime,station_id,temperature,humidity,precipation,precipation_24h
0,2015-01-01 00:00:00,249180010,-3.5,74.0,NaN,NaN
1,2015-01-01 00:10:00,249180010,-3.5,74.0,NaN,NaN
2,2015-01-01 00:20:00,249180010,-3.4,75.0,NaN,NaN
3,2015-01-01 00:30:00,249180010,-3.4,75.0,NaN,NaN
4,2015-01-01 00:40:00,249180010,-3.4,75.0,NaN,NaN
...,...,...,...,...,...,...
89134796,2020-12-31 23:10:00,354220195,-1.8,100.0,0.0,2.6
89134797,2020-12-31 23:20:00,354220195,-1.8,100.0,0.0,2.1
89134798,2020-12-31 23:30:00,354220195,-1.9,100.0,0.0,1.6
89134799,2020-12-31 23:40:00,354220195,-2.0,100.0,0.0,1.1


In [ ]:
df_weather_measurements = df_weather_measurements.merge(df_weather_stations, left_on='station_id', right_on='id', how='left')
df_weather_measurements.drop(columns=['id', 'name'], inplace=True)
df_weather_measurements

In [ ]:
df_weather_measurements['precipation'].isna().sum() / len(df_weather_measurements), df_weather_measurements['precipation_24h'].isna().sum() / len(df_weather_measurements)

(np.float64(0.046114260363648206), np.float64(0.04136308993884039))

In [ ]:
df_weather_measurements.precipation.nlargest(50)

### Estimate weather for each incident

In [28]:
# df_accidents = df_accidents[df_accidents.datetime.dt.year >= 2022]
# df_accidents.sort_values(['datetime'], inplace=True)
# df_accidents.reset_index(inplace=True)
# df_accidents

In [29]:
# https://emf-creaf.github.io/meteolandbook/spatialinterpolation.html

import numpy as np
import pandas as pd
from pykrige.ok import OrdinaryKriging
from pykrige.uk import UniversalKriging
from datetime import datetime
import pyproj
from scipy.interpolate import griddata

# Function to round datetimes to 10-minute intervals
def round_to_tenth_of_hour(dt_series):
    # Convert minute to multiples of 10
    rounded = dt_series.dt.round('10min')
    return rounded

def transform_coordinates(lons, lats, zone=34):
    # Choose appropriate UTM zone for your country - adjust as needed
    utm_proj = pyproj.Proj(proj='utm', zone=zone, ellps='WGS84')
    
    # Transform coordinates
    x, y = utm_proj(lons, lats)
    return x, y

def interpolate_parameter(df_accidents, df_weather, param):
    interpolated_values = []
    
    for idx, row in df_accidents.iterrows():
        if idx % 5000 == 0:
            print(f"{datetime.now()} Processing row {idx} for parameter {param}")
   
        t = row['datetime_10min']
        subset = df_weather[df_weather['datetime'] == t].dropna(subset=[param])
        
        if subset.empty:
            interpolated_values.append(np.nan)
            continue
        
        # Transform coordinates
        lons = subset['longitude'].values
        lats = subset['latitude'].values
        x, y = transform_coordinates(lons, lats)
        vals = subset[param].values
        
        accident_lon = row['longitude']
        accident_lat = row['latitude']
        accident_elevation = row['elevation']
        accident_x, accident_y = transform_coordinates([accident_lon], [accident_lat])

        try:
            if np.all(vals == vals[0]):
                interpolated_values.append(vals[0])
                continue

            UK = UniversalKriging(
                x, y, vals,
                variogram_model='spherical' if 'precipation' in param else 'linear',
                drift_terms=['specified'],
                specified_drift=[subset['elevation'].values.reshape(-1, 1)],
            )
            
            zstar, ss = UK.execute(
                style='points',
                xpoints=np.array([accident_x[0]]),
                ypoints=np.array([accident_y[0]]),
                specified_drift_arrays=[np.array([accident_elevation])],
            )
            value = zstar[0]
            variance = ss[0] # check this

            # Apply basic constraints
            if param in ('precipation', 'precipation_24h'):
                value = max(0, value)
            elif param == 'humidity':
                value = max(0, min(100, value))
            
            interpolated_values.append(value)
            
        except Exception as e:
            print(f"Error interpolating {param} at row {idx}: {e}")
            interpolated_values.append(np.nan)
    
    return interpolated_values

# df_accidents.loc[:, 'datetime_10min'] = round_to_tenth_of_hour(df_accidents['datetime'])

# # Interpolate each parameter
weather_params = ['temperature', 'humidity', 'precipation', 'precipation_24h']

# for param in weather_params:
#     df_accidents.loc[:, param] = interpolate_parameter(df_accidents, df_weather_measurements, param)

# # Drop the intermediate rounding column if desired
# df_accidents.drop(columns=['datetime_10min'], inplace=True)
# df_accidents

In [ ]:
# Create batched processing function
def process_batch(df_batch, df_weather_measurements, weather_params, year, batch_num):
    print(f"{datetime.now()} Processing batch {batch_num} for year {year}")
    
    # Round datetime for the batch
    df_batch.loc[:, 'datetime_10min'] = round_to_tenth_of_hour(df_batch['datetime'])
    
    # Interpolate each parameter
    for param in weather_params:
        print(f"{datetime.now()} Interpolating {param} for batch {batch_num}")
        df_batch.loc[:, param] = interpolate_parameter(df_batch, df_weather_measurements, param)
    
    # Drop the intermediate rounding column
    df_batch.drop(columns=['datetime_10min'], inplace=True)
    
    # Save batch
    output_path = f'../../data/intermediate/batches/incidents_weather_{year}_batch_{batch_num:03d}.csv'
    df_batch.to_csv(output_path, index=False)
    print(f"Saved batch to {output_path}")
    
    return df_batch

# Create directory for batches if it doesn't exist
import os
os.makedirs('../../data/intermediate/batches', exist_ok=True)

# Process each year separately
for year in range(2020, 2018, -1):
    print(f"\nProcessing year {year}")
    
    # Filter data for current year
    year_data = df_accidents[df_accidents.datetime.dt.year == year].copy()
    print(f"Total records for year {year}: {len(year_data)}")
    
    # Process in batches of 1000
    batch_size = 1000
    num_batches = (len(year_data) + batch_size - 1) // batch_size  # Round up division
    
    for batch_num in range(num_batches):
        if os.path.exists(f'../../data/intermediate/batches/incidents_weather_{year}_batch_{batch_num + 1:03d}.csv'):
            continue
        
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(year_data))
        
        batch_df = year_data.iloc[start_idx:end_idx].copy()
        
        # Process the batch
        processed_batch = process_batch(
            batch_df, 
            df_weather_measurements, 
            weather_params,
            year, 
            batch_num + 1
        )
        
        print(f"Completed batch {batch_num + 1}/{num_batches} for year {year}")

print("\nAll batches processed!")

In [ ]:
# df_accidents.to_csv('../../data/intermediate/incidents_with_weather_22_23_4.csv', index=False)

In [6]:
import pandas as pd

df1 = pd.read_csv('../../data/intermediate/incidents_with_weather_22_23_0.csv')
df1

/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_1069/283569286.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('../../data/intermediate/incidents_with_weather_22_23_0.csv')


,id,datetime,longitude,latitude,voivodeship,county,municipality,city,incident_type,other_cause,...,surface_condition,surface_marking,intersection_type,traffic_lights,built_up_area,elevation,temperature,humidity,precipation,precipation_24h
0,108526474,2020-01-01 00:15:00,16.761389,52.514028,wielkopolskie,poznański,Rokietnica,Rokietnica,Najechanie na pieszego,Brak,...,Sucha,Jest,Brak,Brak,Obszar zabudowany,94.348175,3.119401,89.362632,0.000000,0.054164
1,108526534,2020-01-01 00:20:00,18.953917,51.895527,łódzkie,poddębicki,Poddębice,Poddębice,Najechanie na pieszego,Brak,...,Sucha,Nie ma,Brak,Brak,Obszar zabudowany,126.398180,2.962892,89.244036,0.000000,2.809431
2,108523125,2020-01-01 00:25:00,14.568027,53.435666,zachodniopomorskie,Szczecin,Szczecin,Szczecin,Zderzenie pojazdów boczne,Brak,...,Sucha,Jest,Z drogą z pierwsz.,"Jest, działa",Obszar zabudowany,20.107603,3.481939,92.504328,0.000000,0.030858
3,108523133,2020-01-01 00:55:00,20.929499,53.849416,warmińsko-mazurskie,olsztyński,Biskupiec,Rzeck,Najechanie na drzewo,Brak,...,Sucha,Nie ma,Brak,Brak,Obszar zabudowany,149.142600,2.608247,84.542313,0.000000,0.000000
4,108526549,2020-01-01 01:20:00,23.047500,53.405555,podlaskie,moniecki,Jasionówka,Brak w danych źródłowych,Wywrócenie się pojazdu,Brak,...,"Oblodzona, zaśnieżona",Jest,Brak,Brak,Obszar zabudowany,139.611330,2.573578,84.715401,0.000000,9.085860
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88602,125862934,2023-12-31 21:39:00,16.966860,50.850805,dolnośląskie,strzeliński,Borów,Głownin,Najechanie na drzewo,NaN,...,NaN,NaN,NaN,NaN,NaN,158.893660,4.128332,87.037675,0.000000,0.000000
88603,125860432,2023-12-31 22:20:00,19.934027,50.074999,małopolskie,Kraków,Kraków,Kraków,Najechanie na pojazd unieruchomiony,NaN,...,NaN,NaN,NaN,NaN,NaN,222.753650,4.402506,89.314054,0.000010,0.000000
88604,125862027,2023-12-31 23:16:00,20.519268,52.259491,mazowieckie,warszawski zachodni,Kampinos,Wiejca,Najechanie na drzewo,NaN,...,NaN,NaN,NaN,NaN,NaN,88.414474,2.953660,95.485406,0.085774,0.249052
88605,125860445,2023-12-31 23:26:00,17.918390,50.673988,opolskie,Opole,Opole,Opole,Zderzenie pojazdów boczne,NaN,...,NaN,NaN,NaN,NaN,NaN,158.940050,3.744513,90.298512,0.000000,0.000000


In [ ]:
import os

for file in sorted(os.listdir('../../data/intermediate/batches')):
    if not file.endswith('.csv') or 'incidents_weather' not in file or '2020' in file or '2021' in file:
        continue
    df_temp = pd.read_csv(f'../../data/intermediate/batches/{file}')
    print(f"{df_temp.precipation_24h.isna().sum()} {file}")

    df1 = pd.concat([df1, df_temp], ignore_index=True)

89 incidents_weather_2015_batch_001.csv
96 incidents_weather_2015_batch_002.csv
95 incidents_weather_2015_batch_003.csv
92 incidents_weather_2015_batch_004.csv
106 incidents_weather_2015_batch_005.csv
78 incidents_weather_2015_batch_006.csv
99 incidents_weather_2015_batch_007.csv
100 incidents_weather_2015_batch_008.csv
90 incidents_weather_2015_batch_009.csv
104 incidents_weather_2015_batch_010.csv
93 incidents_weather_2015_batch_011.csv
94 incidents_weather_2015_batch_012.csv
74 incidents_weather_2015_batch_013.csv
94 incidents_weather_2015_batch_014.csv
114 incidents_weather_2015_batch_015.csv
109 incidents_weather_2015_batch_016.csv
84 incidents_weather_2015_batch_017.csv
88 incidents_weather_2015_batch_018.csv
93 incidents_weather_2015_batch_019.csv
83 incidents_weather_2015_batch_020.csv
92 incidents_weather_2015_batch_021.csv
94 incidents_weather_2015_batch_022.csv
100 incidents_weather_2015_batch_023.csv
102 incidents_weather_2015_batch_024.csv
100 incidents_weather_2015_batch_

In [14]:
df1.sort_values(['datetime'], inplace=True)
df1

,id,datetime,longitude,latitude,voivodeship,county,municipality,city,incident_type,other_cause,...,surface_marking,intersection_type,traffic_lights,built_up_area,elevation,datetime_10min,temperature,humidity,precipation,precipation_24h
235076,98710099,2015-01-01 00:30:00,20.963028,51.647694,mazowieckie,białobrzeski,Białobrzegi,Pohulanka,Najechanie na zwierzę,"Obiekty, zwierzęta na drodze",...,Jest,Brak,Brak,Obszar zabudowany,123.730960,NaN,-0.736069,94.641313,0.002010,0.000000
226386,98710983,2015-01-01 00:50:00,18.595278,54.416389,pomorskie,Gdańsk,Gdańsk,Gdańsk,Najechanie na pieszego,Brak,...,Jest,Brak,Brak,Obszar zabudowany,8.724400,NaN,5.547733,94.465185,0.000245,0.002012
212334,98709614,2015-01-01 00:50:00,22.030278,49.509222,podkarpackie,sanocki,Bukowsko,Nowotaniec,Najechanie na pieszego,Brak,...,Nie ma,Brak,Brak,Obszar zabudowany,386.764040,NaN,-6.320154,68.281133,0.000000,0.004419
54556,98708579,2015-01-01 00:57:00,21.396972,50.947500,świętokrzyskie,ostrowiecki,Ostrowiec Świętokrzyski,Ostrowiec Świętokrzyski,Zderzenie pojazdów boczne,Brak,...,Jest,Z drogą z pierwsz.,"Jest, nie działa",Obszar zabudowany,196.577650,NaN,-2.609348,89.742499,0.000000,0.000000
235862,98709800,2015-01-01 01:00:00,16.591861,51.846722,wielkopolskie,Leszno,Leszno,Leszno,Zderzenie pojazdów boczne,Brak,...,Jest,Z drogą z pierwsz.,"Jest, nie działa",Obszar zabudowany,96.276580,NaN,1.702085,97.810005,0.075788,0.087369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42247,125862934,2023-12-31 21:39:00,16.966860,50.850805,dolnośląskie,strzeliński,Borów,Głownin,Najechanie na drzewo,NaN,...,NaN,NaN,NaN,NaN,158.893660,2023-12-31 21:40:00,4.128332,87.037675,NaN,NaN
42248,125860432,2023-12-31 22:20:00,19.934027,50.074999,małopolskie,Kraków,Kraków,Kraków,Najechanie na pojazd unieruchomiony,NaN,...,NaN,NaN,NaN,NaN,222.753650,2023-12-31 22:20:00,4.402506,89.314054,NaN,NaN
42249,125862027,2023-12-31 23:16:00,20.519268,52.259491,mazowieckie,warszawski zachodni,Kampinos,Wiejca,Najechanie na drzewo,NaN,...,NaN,NaN,NaN,NaN,88.414474,2023-12-31 23:20:00,2.953660,95.485406,NaN,NaN
42250,125860445,2023-12-31 23:26:00,17.918390,50.673988,opolskie,Opole,Opole,Opole,Zderzenie pojazdów boczne,NaN,...,NaN,NaN,NaN,NaN,158.940050,2023-12-31 23:30:00,3.744513,90.298512,NaN,NaN


In [15]:
df1.to_csv('../../data/intermediate/incidents_with_weather.csv', index=False)

In [18]:
# interpolate_parameter(df_accidents.loc[634:634], df_weather_measurements, 'precipation')

In [27]:
df_accidents = df1
df_accidents['datetime'] = pd.to_datetime(df_accidents['datetime'])

## Add socioeconomic data

In [29]:
df_socioeconomic = pd.read_csv('../../data/intermediate/socioeconomic.csv')
df_socioeconomic['id'] = df_socioeconomic['id'].astype(str).str.zfill(7)
df_socioeconomic.tail()

,id,type,name,year,Indicators: population per 1 km2 (person),Road types: paved surface (km),Road types: improved paved surface (km),Vehicle types: motor vehicles and tractors (units),Vehicle types: passenger cars (units),Vehicle types: total buses (units),Vehicle types: trucks (units),Vehicle types: semi-trailer tractors (units),Road accidents: total accidents (units),Road accidents: total casualties (person),Road accidents: fatalities (person),Road accidents: injured (person),Road types: municipal and county paved roads per 100km2 (km),Road types: municipal and county paved roads per 10.000 inhabitants (km)
11558,3263000,county,powiat m. Świnoujście,2019,202.3,61.5,55.0,27332.0,22611.0,137.0,2569.0,117.0,34.0,NaN,3.0,36.0,49.0,24.2
11559,3263000,county,powiat m. Świnoujście,2020,199.6,63.8,58.5,28192.0,23311.0,138.0,2647.0,116.0,27.0,NaN,2.0,26.0,50.1,25.1
11560,3263000,county,powiat m. Świnoujście,2021,197.1,63.0,58.9,28783.0,23755.0,131.0,2689.0,120.0,24.0,NaN,3.0,22.0,50.4,25.6
11561,3263000,county,powiat m. Świnoujście,2022,194.8,66.6,62.6,29587.0,24336.0,136.0,2796.0,117.0,19.0,NaN,0.0,22.0,52.2,26.8
11562,3263000,county,powiat m. Świnoujście,2023,192.5,65.9,62.0,30165.0,24763.0,138.0,2864.0,126.0,29.0,32.0,1.0,31.0,56.8,29.5


In [30]:
df_counties_soc = df_socioeconomic[df_socioeconomic['type'] == 'county']

# https://bdl.stat.gov.pl/bdl/metadane/teryt/lista
voivodeship_codes = {
    'dolnośląskie': '0200000',
    'kujawsko-pomorskie': '0400000',
    'lubelskie': '0600000',
    'lubuskie': '0800000',
    'łódzkie': '1000000',
    'małopolskie': '1200000',
    'mazowieckie': '1400000',
    'opolskie': '1600000',
    'podkarpackie': '1800000',
    'podlaskie': '2000000',
    'pomorskie': '2200000',
    'śląskie': '2400000',
    'świętokrzyskie': '2600000',
    'warmińsko-mazurskie': '2800000',
    'wielkopolskie': '3000000',
    'zachodniopomorskie': '3200000'
}

code_to_voivodeship = {code[:2]: name for name, code in voivodeship_codes.items()}
df_counties_soc = df_counties_soc.assign(
    voivodeship=lambda x: x['id'].astype('str').str[:2].map(code_to_voivodeship)
)
df_counties_soc

,id,type,name,year,Indicators: population per 1 km2 (person),Road types: paved surface (km),Road types: improved paved surface (km),Vehicle types: motor vehicles and tractors (units),Vehicle types: passenger cars (units),Vehicle types: total buses (units),Vehicle types: trucks (units),Vehicle types: semi-trailer tractors (units),Road accidents: total accidents (units),Road accidents: total casualties (person),Road accidents: fatalities (person),Road accidents: injured (person),Road types: municipal and county paved roads per 100km2 (km),Road types: municipal and county paved roads per 10.000 inhabitants (km),voivodeship
58,0201000,county,powiat bolesławiecki,1995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dolnośląskie
59,0201000,county,powiat bolesławiecki,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dolnośląskie
60,0201000,county,powiat bolesławiecki,1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dolnośląskie
61,0201000,county,powiat bolesławiecki,1998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dolnośląskie
62,0201000,county,powiat bolesławiecki,1999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dolnośląskie
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11558,3263000,county,powiat m. Świnoujście,2019,202.3,61.5,55.0,27332.0,22611.0,137.0,2569.0,117.0,34.0,NaN,3.0,36.0,49.0,24.2,zachodniopomorskie
11559,3263000,county,powiat m. Świnoujście,2020,199.6,63.8,58.5,28192.0,23311.0,138.0,2647.0,116.0,27.0,NaN,2.0,26.0,50.1,25.1,zachodniopomorskie
11560,3263000,county,powiat m. Świnoujście,2021,197.1,63.0,58.9,28783.0,23755.0,131.0,2689.0,120.0,24.0,NaN,3.0,22.0,50.4,25.6,zachodniopomorskie
11561,3263000,county,powiat m. Świnoujście,2022,194.8,66.6,62.6,29587.0,24336.0,136.0,2796.0,117.0,19.0,NaN,0.0,22.0,52.2,26.8,zachodniopomorskie


In [31]:
df_counties_soc['name'] =  df_counties_soc['name'].str.replace('powiat ', '')
df_counties_soc.drop(columns=['id'], inplace=True)

df_accidents.loc[:, 'year'] = df_accidents['datetime'].dt.year

In [32]:
df_merged = df_accidents.merge(df_counties_soc, left_on=['county', 'voivodeship', 'year'], right_on=['name', 'voivodeship', 'year'], how='left')
df_merged.drop(columns=['year', 'type', 'name'], inplace=True)
df_merged.head()

,id,datetime,longitude,latitude,voivodeship,county,municipality,city,incident_type,other_cause,...,Vehicle types: passenger cars (units),Vehicle types: total buses (units),Vehicle types: trucks (units),Vehicle types: semi-trailer tractors (units),Road accidents: total accidents (units),Road accidents: total casualties (person),Road accidents: fatalities (person),Road accidents: injured (person),Road types: municipal and county paved roads per 100km2 (km),Road types: municipal and county paved roads per 10.000 inhabitants (km)
0,98710099,2015-01-01 00:30:00,20.963028,51.647694,mazowieckie,białobrzeski,Białobrzegi,Pohulanka,Najechanie na zwierzę,"Obiekty, zwierzęta na drodze",...,22616.0,115.0,6654.0,283.0,34.0,NaN,7.0,40.0,80.9,154.0
1,98710983,2015-01-01 00:50:00,18.595278,54.416389,pomorskie,Gdańsk,Gdańsk,Gdańsk,Najechanie na pieszego,Brak,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,98709614,2015-01-01 00:50:00,22.030278,49.509222,podkarpackie,sanocki,Bukowsko,Nowotaniec,Najechanie na pieszego,Brak,...,42679.0,439.0,6279.0,339.0,29.0,NaN,8.0,28.0,46.6,59.6
3,98708579,2015-01-01 00:57:00,21.396972,50.947500,świętokrzyskie,ostrowiecki,Ostrowiec Świętokrzyski,Ostrowiec Świętokrzyski,Zderzenie pojazdów boczne,Brak,...,46503.0,274.0,5757.0,506.0,48.0,NaN,0.0,59.0,112.6,61.8
4,98709800,2015-01-01 01:00:00,16.591861,51.846722,wielkopolskie,Leszno,Leszno,Leszno,Zderzenie pojazdów boczne,Brak,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
df_merged

,id,datetime,longitude,latitude,voivodeship,county,municipality,city,incident_type,other_cause,...,Vehicle types: passenger cars (units),Vehicle types: total buses (units),Vehicle types: trucks (units),Vehicle types: semi-trailer tractors (units),Road accidents: total accidents (units),Road accidents: total casualties (person),Road accidents: fatalities (person),Road accidents: injured (person),Road types: municipal and county paved roads per 100km2 (km),Road types: municipal and county paved roads per 10.000 inhabitants (km)
0,98710099,2015-01-01 00:30:00,20.963028,51.647694,mazowieckie,białobrzeski,Białobrzegi,Pohulanka,Najechanie na zwierzę,"Obiekty, zwierzęta na drodze",...,22616.0,115.0,6654.0,283.0,34.0,NaN,7.0,40.0,80.9,154.0
1,98710983,2015-01-01 00:50:00,18.595278,54.416389,pomorskie,Gdańsk,Gdańsk,Gdańsk,Najechanie na pieszego,Brak,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,98709614,2015-01-01 00:50:00,22.030278,49.509222,podkarpackie,sanocki,Bukowsko,Nowotaniec,Najechanie na pieszego,Brak,...,42679.0,439.0,6279.0,339.0,29.0,NaN,8.0,28.0,46.6,59.6
3,98708579,2015-01-01 00:57:00,21.396972,50.947500,świętokrzyskie,ostrowiecki,Ostrowiec Świętokrzyski,Ostrowiec Świętokrzyski,Zderzenie pojazdów boczne,Brak,...,46503.0,274.0,5757.0,506.0,48.0,NaN,0.0,59.0,112.6,61.8
4,98709800,2015-01-01 01:00:00,16.591861,51.846722,wielkopolskie,Leszno,Leszno,Leszno,Zderzenie pojazdów boczne,Brak,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249952,125862934,2023-12-31 21:39:00,16.966860,50.850805,dolnośląskie,strzeliński,Borów,Głownin,Najechanie na drzewo,NaN,...,36762.0,325.0,4419.0,421.0,23.0,27.0,4.0,23.0,79.1,117.7
249953,125860432,2023-12-31 22:20:00,19.934027,50.074999,małopolskie,Kraków,Kraków,Kraków,Najechanie na pojazd unieruchomiony,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
249954,125862027,2023-12-31 23:16:00,20.519268,52.259491,mazowieckie,warszawski zachodni,Kampinos,Wiejca,Najechanie na drzewo,NaN,...,130743.0,482.0,25941.0,5565.0,48.0,59.0,9.0,50.0,128.8,50.8
249955,125860445,2023-12-31 23:26:00,17.918390,50.673988,opolskie,Opole,Opole,Opole,Zderzenie pojazdów boczne,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
df_merged.columns = (
    df_merged.columns
      .str.replace(r"\(.*?\)", "", regex=True)  # remove any parentheses and text inside them
      .str.lower()                              # convert to lowercase
      .str.strip()                              # remove whitespace from start and end
      .str.replace(":", "")                     # replace :
      .str.replace(" ", "_")                    # replace spaces with underscores
)

df_merged.columns

Index(['id', 'datetime', 'longitude', 'latitude', 'voivodeship', 'county',
       'municipality', 'city', 'incident_type', 'other_cause',
       'driver_under_influence', 'site_characteristic', 'speed_limit',
       'road_type', 'road_geometry', 'pavement', 'surface_condition',
       'surface_marking', 'intersection_type', 'traffic_lights',
       'built_up_area', 'elevation', 'temperature', 'humidity', 'precipation',
       'precipation_24h', 'indicators_population_per_1_km2',
       'road_types_paved_surface', 'road_types_improved_paved_surface',
       'vehicle_types_motor_vehicles_and_tractors',
       'vehicle_types_passenger_cars', 'vehicle_types_total_buses',
       'vehicle_types_trucks', 'vehicle_types_semi-trailer_tractors',
       'road_accidents_total_accidents', 'road_accidents_total_casualties',
       'road_accidents_fatalities', 'road_accidents_injured',
       'road_types_municipal_and_county_paved_roads_per_100km2',
       'road_types_municipal_and_county_paved_roads

In [40]:
df_merged.to_csv('../../data/intermediate/data_all.csv', index=False)